# Musicm8 v10.1 — resilient learned producer

Musicm8 now uses the learned MIDI-LLM composer **section by section**, with targeted AI role rescue when a section is weak on bass, harmony, melody or drums. The audible notes are then put on one shared groove clock, harmony-checked, rendered with sampled instruments, and the lead vocal is generated from explicit words/phonemes plus the actual song score.

The first music run downloads the MIDI-LLM weights to Drive for reuse. SoulX uses its own isolated Python environment and is cached separately.

In [ ]:
# ============================================================
# MUSICM8 V10.1 — ONE CLICK COMPLETE SONG
# ============================================================
import os, sys, json, shutil, subprocess, secrets
from pathlib import Path

IDEA = "dark UK garage song about knowing a relationship is over but not being able to leave, emotional chords, deep moving bass"
BARS = 32
FIXED_SEED = None  # put a previous seed here to reproduce it
SEED = int(FIXED_SEED) if FIXED_SEED is not None else secrets.randbelow(2_000_000_000)
print(f"🎲 MUSICM8 SONG SEED: {SEED}")

# Attempts per section. Weak roles can also trigger targeted AI rescue attempts.
COMPOSER_CANDIDATES = 2

# Paste your own exact lyrics here. Leave blank for AI-written lyrics.
CUSTOM_LYRICS = r"""
""".strip()

# Optional clean voice/timbre reference you own or have permission to use.
VOICE_REFERENCE = ""
VOCALS = True
VOCAL_STEPS = 24
AI_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"

from google.colab import drive
drive.mount("/content/drive", force_remount=False)
ROOT = Path("/content/drive/MyDrive/Musicm8")
WORK = ROOT / "work"
AUDIO = ROOT / "audio"
REPO = Path("/content/Musicm8")
REPO_URL = "https://github.com/Elephant-logic/Musicm8.git"
PROJECT = WORK / "ai_projects/latest"
WORK.mkdir(parents=True, exist_ok=True); AUDIO.mkdir(parents=True, exist_ok=True)

if (REPO / ".git").exists():
    subprocess.run(["git","-C",str(REPO),"fetch","--depth","1","origin","main"], check=True)
    subprocess.run(["git","-C",str(REPO),"reset","--hard","origin/main"], check=True)
else:
    shutil.rmtree(REPO, ignore_errors=True)
    subprocess.run(["git","clone","--depth","1",REPO_URL,str(REPO)], check=True)
commit = subprocess.check_output(["git","-C",str(REPO),"rev-parse","--short","HEAD"], text=True).strip()
print("📦 Musicm8 commit:", commit)

os.chdir(REPO)
subprocess.run([sys.executable,"-m","pip","install","-q","-r","requirements-ai.txt"], check=True)
subprocess.run(["apt-get","update","-qq"], check=True)
subprocess.run(["apt-get","install","-y","-qq","ffmpeg","fluidsynth","fluid-soundfont-gm"], check=False)
subprocess.run(["apt-get","install","-y","-qq","musescore-general-soundfont"], check=False)

import torch
if not torch.cuda.is_available(): raise RuntimeError("No GPU connected. Runtime → Change runtime type → GPU.")
print("GPU:", torch.cuda.get_device_name(0))
print("GPU VRAM: %.1f GB" % (torch.cuda.get_device_properties(0).total_memory / 1024**3))

lyrics_input = WORK / "user_lyrics_input.txt"
if CUSTOM_LYRICS.strip():
    lyrics_input.write_text(CUSTOM_LYRICS.strip()+"\n", encoding="utf-8")
    print("✍️ USER LYRICS MODE — exact words preserved")
else:
    lyrics_input.unlink(missing_ok=True)
    print("✍️ AI LYRICS MODE")

cmd = [
    sys.executable,"-u","ai_producer_workflow.py",
    "--root",str(ROOT),"--repo",str(REPO),
    "--idea",IDEA,"--bars",str(BARS),"--seed",str(SEED),
    "--ai-model",AI_MODEL,"--composer-candidates",str(COMPOSER_CANDIDATES),
    "--vocal-steps",str(VOCAL_STEPS)
]
if CUSTOM_LYRICS.strip(): cmd += ["--lyrics-file",str(lyrics_input)]
if VOICE_REFERENCE.strip(): cmd += ["--voice-reference",VOICE_REFERENCE.strip()]
if not VOCALS: cmd.append("--no-vocals")

proc = subprocess.run(cmd, check=False)
if proc.returncode != 0:
    print("\n================ CURRENT RUN FAILURE ================")
    for p in [PROJECT/"workflow_failure.json", PROJECT/"composition_failure.json"]:
        if p.exists():
            print(f"\n--- {p.name} ---\n{p.read_text(errors='ignore')}")
    log = PROJECT/"vocals/vocal_backend.log"
    if log.exists():
        print("\n--- SOULX BACKEND LOG TAIL ---\n" + "\n".join(log.read_text(errors="ignore").splitlines()[-120:]))
    print("=====================================================")
    raise RuntimeError(f"Musicm8 stopped. See CURRENT RUN FAILURE above. Exit code {proc.returncode}")

PROJECT.mkdir(parents=True, exist_ok=True)
(PROJECT/"song_seed.txt").write_text(str(SEED), encoding="utf-8")

lf = PROJECT/"lyrics.txt"
print("\n================ 📝 LYRICS USED ================\n")
print(lf.read_text(encoding="utf-8") if lf.exists() else "No lyrics file")
print("=================================================\n")

cr = PROJECT/"composition_report.json"
if cr.exists():
    c = json.loads(cr.read_text())
    print("🧠 LEARNED SECTION COMPOSER")
    print("model:", c.get("composer"))
    print("mode:", c.get("composition_mode"))
    print("role notes:", c.get("role_notes"))
    print("sections:", len(c.get("sections", [])))
    print("rule-note fallback:", c.get("rule_note_composer_fallback"))
    for s in c.get("sections", []):
        print("  •", s.get("name"), s.get("final_role_notes"), "warnings=", s.get("quality_warnings", []))

status_path = PROJECT/"final_status.json"
status = json.loads(status_path.read_text()) if status_path.exists() else {}
print("\n🎯 FINAL STATUS", status.get("final_kind"))

from IPython.display import Audio, display
for label,p in [
    ("AI COMPOSED INSTRUMENTAL",PROJECT/"master_instrumental.wav"),
    ("SOULX SCORE GUIDE",PROJECT/"vocals/soulx_score_guide.wav"),
    ("QA-PASSED LEAD",PROJECT/"vocals/neural_lead_synced.wav")
]:
    if p.exists():
        print("\n🎵",label); display(Audio(str(p)))
final = PROJECT/"master.wav"
if status.get("final_kind") == "song_with_vocals" and final.exists():
    print("\n✅ FINAL SONG — VOCALS PASSED"); display(Audio(str(final)))
elif VOCALS:
    print("\n❌ VOCAL FINAL FAILED — instrumental safety copy only")
    if final.exists(): display(Audio(str(final)))
    log = PROJECT/"vocals/vocal_backend.log"
    if log.exists(): print("\n--- VOCAL BACKEND LOG TAIL ---\n" + "\n".join(log.read_text(errors="ignore").splitlines()[-120:]))
else:
    print("\n✅ FINAL INSTRUMENTAL")
    if final.exists(): display(Audio(str(final)))
print(f"\n🌱 Seed: {SEED}")

## 🎤 Vocal-only retry

Run this only after the current instrumental is worth keeping. It leaves the music untouched and regenerates the score-controlled singer against the current chord and lead MIDI.

In [ ]:
# MUSICM8 V10.1 — VOCAL ONLY
import os, sys, subprocess
from pathlib import Path
from IPython.display import Audio, display
ROOT=Path('/content/drive/MyDrive/Musicm8'); REPO=Path('/content/Musicm8'); PROJECT=ROOT/'work/ai_projects/latest'
VOICE_REFERENCE=""
VOCAL_STEPS=24
seed_file=PROJECT/'song_seed.txt'; SEED=int(seed_file.read_text().strip()) if seed_file.exists() else 42
subprocess.run(['git','-C',str(REPO),'fetch','--depth','1','origin','main'],check=True)
subprocess.run(['git','-C',str(REPO),'reset','--hard','origin/main'],check=True)
os.chdir(REPO)
cmd=[sys.executable,'-u','retry_vocals_v10.py','--root',str(ROOT),'--repo',str(REPO),'--steps',str(VOCAL_STEPS),'--seed',str(SEED)]
if VOICE_REFERENCE.strip(): cmd += ['--voice-reference',VOICE_REFERENCE.strip()]
proc=subprocess.run(cmd,check=False)
if proc.returncode != 0:
    log=PROJECT/'vocals/vocal_backend.log'
    if log.exists(): print('\n--- SOULX BACKEND LOG TAIL ---\n'+'\n'.join(log.read_text(errors='ignore').splitlines()[-160:]))
    raise RuntimeError(f'Vocal retry failed with exit code {proc.returncode}')
print('\n================ 📝 LYRICS USED ================\n'+(PROJECT/'lyrics.txt').read_text()+'=================================================\n')
for label,p in [('SOULX GUIDE',PROJECT/'vocals/soulx_score_guide.wav'),('QA-PASSED LEAD',PROJECT/'vocals/neural_lead_synced.wav'),('FINAL SONG',PROJECT/'master.wav')]:
    if p.exists(): print('\n🎵',label); display(Audio(str(p)))

## Diagnostics

This prints only the current run's useful failure/status files and the SoulX backend tail.

In [ ]:
from pathlib import Path
project=Path('/content/drive/MyDrive/Musicm8/work/ai_projects/latest')
for p in [project/'workflow_failure.json',project/'composition_failure.json',project/'composition_report.json',project/'render_timing_report.json',project/'soundfont_render_report.json',project/'final_status.json',project/'vocals/vocal_status.json',project/'vocals/vocal_quality.json',project/'vocals/vocal_word_quality.json']:
    if p.exists(): print('\n---',p.name,'---\n'+p.read_text())
log=project/'vocals/vocal_backend.log'
if log.exists(): print('\n--- SOULX BACKEND LOG TAIL ---\n'+'\n'.join(log.read_text(errors='ignore').splitlines()[-180:]))